# Resource-Aware and Noise-Realized Operator Selection in ADAPT-VQE

**Benchmark on H2 and LiH under calibrated depolarizing noise.**

This notebook reproduces the study cell by cell.

- Sections 1-5 **run live** (~15-20 min total on a CPU).
- Section 6 loads the **pre-computed** full sweep (`results/results_novel.csv`, ~2 h to regenerate) and renders the comparison tables and figures.

### How to run

**Google Colab:** put this notebook and the project's `.py` files + `results/` folder in the same place (e.g. upload the whole folder, or mount Drive). Run the **SETUP** cell -- it installs the Qiskit / PennyLane packages -- then **Runtime -> Restart session**, then **Run all**.

**Locally:** place this notebook in the project directory. The packages in `requirements.txt` are already installed, so the pip line is a no-op; just Run all.

In [ ]:
# === SETUP ===  (Colab: run this, then Runtime -> Restart session, then Run all)
import importlib.util, sys
_need = any(importlib.util.find_spec(m) is None for m in
            ('qiskit', 'qiskit_aer', 'qiskit_nature', 'qiskit_ibm_runtime',
             'qiskit_algorithms', 'pennylane'))
if _need:
    !pip -q install "qiskit>=2.3.0" "qiskit-aer>=0.17.2" "qiskit-nature>=0.8.0" "qiskit-algorithms>=0.4.0" "qiskit-ibm-runtime>=0.45.0" "pennylane>=0.45.0" "pennylane-qiskit>=0.45.0" "numpy>=2.0" "scipy>=1.13" "pandas>=2.0" "matplotlib>=3.8"
    print('\n>>> Installed. Now: Runtime -> Restart session, then Run all. <<<')
else:
    print('dependencies present')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import molecules as mol, noise_models as nz, pennylane_resource_aware_adapt as ra
print('imports OK  |  selection rules:', list(ra.STRATEGIES))

## 1. Shared ground truth: molecular Hamiltonians

`molecules.validate()` checks (a) the pure-Python differentiable Hartree-Fock integrals reproduce published full-space FCI, (b) the Qiskit and PennyLane Hamiltonians share a spectrum to ~1e-12 Ha (they are built from identical integrals), (c) HF sits above FCI.

In [ ]:
assert mol.validate()

## 2. Noise model on an equivalent footing

`noise_models.selftest()` builds the SAME explicit circuit in Qiskit and PennyLane, applies each side's depolarizing channels (Kraus operators reproducing Qiskit's convention exactly), and compares <ZZZ>. Agreement to ~1e-10 proves both frameworks are noised identically.

In [ ]:
assert nz.selftest()

## 3. The vacuous-selection problem (diagnostic)

At the Hartree-Fock reference, for each molecule: how many pool operators have a non-zero gradient, what CNOT costs do they span, and which operator does `standard` (score = |grad|) vs `resource-aware` (score = |grad|/(1+lambda*cost)) pick.

**Expected:** single excitations have zero gradient (Brillouin), the gradient-carrying doubles are all equal cost, so the cost denominator never changes the arg-max -- the rule is inert.

In [ ]:
noiseless = nz.make_noise_levels(scales=(0.0,))[0]
for spec in mol.BENCHMARK_MOLECULES:
    pool = ra.build_pool(spec)
    g = ra._candidate_gradients(spec, noiseless, [], [], pool, method='central')
    costs = np.array([o.cnot_cost for o in pool], float)
    nz_costs = sorted({int(c) for c, gi in zip(costs, g) if gi > 1e-8})
    std = int(np.argmax(g)); rac = int(np.argmax(g / (1.0 + 1.0 * costs)))
    print(f'{spec.label:15s} pool={len(pool):3d}  nonzero-grad ops={int((g > 1e-8).sum())}  '
          f'distinct CNOT cost among them={nz_costs}')
    print(f'   standard picks           [{std:2d}] {pool[std].label}')
    print(f'   resource-aware(l=1) picks [{rac:2d}] {pool[rac].label}')
    print(f'   --> SAME PICK: {std == rac}')

## 4. The fix + verification: qubit-ADAPT pool

The pool is switched to individual Jordan-Wigner Pauli-string rotations (CNOT cost `2*(weight-1)`, Z-strings retained), giving a real spread of costs. Verify noiselessly that (a) `resource-aware` now selects a **different** operator sequence from `standard` on LiH, and (b) `lambda=0` still reproduces `standard` **exactly**.

In [ ]:
for spec in mol.BENCHMARK_MOLECULES:
    s = ra.standard_adapt_pennylane(spec, noiseless, max_operators=8, opt_maxiter=100)
    r = ra.resource_aware_adapt(spec, noiseless, lam=1.0, max_operators=8, opt_maxiter=100)
    z = ra.resource_aware_adapt(spec, noiseless, lam=0.0, max_operators=8, opt_maxiter=100)
    print(f'\n{spec.label}')
    print(f"  standard        {s['n_operators']} ops  {s['cnot_count']:2d} CNOT  err {s['energy_error']*1e3:7.3f} mHa")
    print(f"  resource-aware  {r['n_operators']} ops  {r['cnot_count']:2d} CNOT  err {r['energy_error']*1e3:7.3f} mHa")
    print(f"  selection differs from standard? {s['operators'] != r['operators']}")
    print(f"  lambda=0 == standard exactly?  ops_match={z['operators'] == s['operators']}  "
          f"|dE|={abs(z['energy'] - s['energy']):.1e}")

## 5. Novel rule: noise-realized selection (live, LiH at 0.5x device noise)

`noise_realized`: gradient-screen to the top 4, re-optimize each **with the noise channel present**, select by `max(0, realised dE) / (1 + lambda*cost)`. `adaptive_lambda`: lambda is a feedback controller that rises in the noise-dominated tail.

_This cell runs 4 noisy ADAPT loops on 6-qubit LiH; expect ~15-20 min._

In [ ]:
noisy = nz.make_noise_levels(scales=(0.5,))[0]
spec = mol.LIH
runs = [
    ('standard',                     ra.standard_adapt_pennylane,             {}),
    ('resource-aware',               ra.resource_aware_adapt,                 {'lam': 1.0}),
    ('noise-realized',               ra.noise_realized_adapt,                 {'lam': 1.0}),
    ('noise-realized+adaptive-lam',  ra.noise_realized_adapt_adaptive_lambda, {'lam': 1.0}),
]
for name, fn, kw in runs:
    r = fn(spec, noisy, max_operators=6, opt_maxiter=80, **kw)
    li, lf = r.get('lambda_init'), r.get('lambda_final')
    lf = round(float(lf), 2) if lf is not None else lf
    print(f'{name:30s} err {r["energy_error"]*1e3:7.3f} mHa   CNOT {r["cnot_count"]:2d}   '
          f'ops {r["n_operators"]}   lambda {li}->{lf}')

## 6. Full sweep (pre-computed)

The complete benchmark is **2 molecules x 4 noise scales x 7 strategies = 56 runs**, ~2 h, saved in `results/results_novel.csv`. To regenerate:

```bash
python run_experiments.py --skip-validation --out results/results_novel.csv --pl-max-operators 8 --pl-opt-maxiter 100
python analyze_results.py --csv results/results_novel.csv --outdir results/novel
```

In [ ]:
df = pd.read_csv('results/results_novel.csv')
pl = df[df.framework == 'pennylane'].copy()
pl['err_mHa'] = pl.energy_error * 1e3
order = ['ADAPT-VQE(standard,PL)', 'ADAPT-VQE(resource-aware)',
         'ADAPT-VQE(noise-realized)', 'ADAPT-VQE(noise-realized,adaptive-lam)']
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 20)
print('ENERGY ERROR  (mHa)')
print(pl.pivot_table(index=['molecule', 'noise_scale'], columns='strategy', values='err_mHa')[order].round(3))
print('\nCNOT COUNT')
print(pl.pivot_table(index=['molecule', 'noise_scale'], columns='strategy', values='cnot_count')[order].astype('Int64'))

In [ ]:
# Option B baselines (Qiskit) for context
qk = df[df.framework == 'qiskit'].copy(); qk['err_mHa'] = qk.energy_error * 1e3
print(qk.pivot_table(index=['molecule', 'noise_scale'], columns='strategy', values='err_mHa').round(2))

In [ ]:
import os
from IPython.display import Image, display
for p in ['results/novel/energy_error_vs_noise.png', 'results/novel/circuit_depth.png']:
    if os.path.exists(p):
        print(p); display(Image(p))

## 7. Findings

1. **`resource-aware` is a null result.** `|grad|/(1+lambda*cost)` selects the identical ansatz to standard ADAPT at every non-zero noise level for both molecules (delta = 0.000 mHa). Cause: equal-cost gradient-carrying doubles + zero single-excitation gradients at HF; where the rule *does* diverge (operators 6-8, noiseless) the ansatz has already reached FCI. `lambda=0` reproduces standard ADAPT exactly.

2. **`noise-realized` selection works.** Scoring by the energy an operator *actually* buys under the noise channel (not its noise-blind gradient) beats both standard and resource-aware ADAPT at every intermediate noise level -- H2 0.25x: 13.83 vs 15.26 mHa; LiH 0.25x / 0.5x: -0.46 / -0.90 mHa -- **at identical circuit cost**. Noiselessly it reaches chemical accuracy with 26 CNOTs vs 36. The gain is from picking the specific Pauli string whose contribution survives decoherence.

3. **`adaptive-lambda`** tracks the noise regime correctly (lambda_final ~0.4 at low noise, ~2.5 at the device rate) but the minimal-basis pools of H2 / LiH(2e,3o) lack the cheap alternative operators it would need to change a selection.

Full write-ups: `RESULTS.md`, `NOVEL_RESULTS.md`, `IMPROVEMENTS.md`.